In [1]:
import importlib
import fetch_data as td
import model as mod

In [2]:
# Après avoir modifié le fichier .py :
importlib.reload(mod)

<module 'model' from '/home/onyxia/MiseEnProd/model.py'>

In [3]:
ids = td.get_movie_ids_list(td.HEADERS, 10, ending_date="03-18-2026", ascending=False)
historic_df = td.get_movies_info(ids, td.HEADERS)
df_cleaned = td.clean_data(historic_df)
df_cleaned.head()

Fetching movie IDs...


100%|██████████| 10/10 [00:00<00:00, 12.23it/s]


Fetching movie details...


100%|██████████| 200/200 [00:31<00:00,  6.34it/s]


,budget,id,origin_country,overview,popularity,release_date,revenue,runtime,title,vote_average,vote_count,belongs_to_collection,main_genre_id,main_genre_name,full_poster_path,title_char_count,overview_char_count,timestamp
0,110000000,1226863,US,Having thwarted Bowser's previous plot to marr...,298.6540,2026-04-01,122100000,98,The Super Mario Galaxy Movie,6.981,157,NaN,12,Adventure,https://image.tmdb.org/t/p/original/eJGWx219Zc...,28,310,1775001600
1,200000000,687163,US,Science teacher Ryland Grace wakes up on a spa...,237.9095,2026-03-15,323427010,157,Project Hail Mary,8.185,957,NaN,878,Science Fiction,https://image.tmdb.org/t/p/original/yihdXomYb5...,17,433,1773532800
2,0,1115544,US,Two gangsters and the woman they love try to s...,349.1562,2026-03-14,0,107,Mike & Nick & Nick & Alice,6.800,149,NaN,35,Comedy,https://image.tmdb.org/t/p/original/7F0jc75HrS...,26,181,1773446400
3,0,1084187,US,A troupe of ballerinas find themselves fightin...,265.0209,2026-03-13,0,90,Pretty Lethal,6.967,198,NaN,28,Action,https://image.tmdb.org/t/p/original/znTPnXCK3l...,13,167,1773360000
4,0,875828,GB,After his estranged son gets embroiled in a Na...,129.9931,2026-03-05,0,112,Peaky Blinders: The Immortal Man,7.292,644,NaN,80,Crime,https://image.tmdb.org/t/p/original/gRMalasZEz...,32,151,1772668800


In [4]:
df_train = df_cleaned[df_cleaned["revenue"] > 0]
y = df_train["revenue"]

In [5]:
pipeline = mod.create_random_forest_pipeline()
scores = mod.model_cross_validation(df_train, pipeline)

In [6]:
print("Moyenne du revenue    :", y.mean())
print("Écart-type du revenue :", y.std())
print("Médiane du revenue    :", y.median())

# RMSE relatif (coefficient de variation)
rmse_moyen = -scores.mean()
print("RMSE :", rmse_moyen)
print("RMSE relatif (RMSE / std)  :", rmse_moyen / y.std())
print("RMSE relatif (RMSE / mean) :", rmse_moyen / y.mean())

Moyenne du revenue    : 95432427.5042735
Écart-type du revenue : 233702218.6921399
Médiane du revenue    : 27834512.0
RMSE : 147895350.47091815
RMSE relatif (RMSE / std)  : 0.6328367411254376
RMSE relatif (RMSE / mean) : 1.5497389549720437


In [7]:
param_grid = {
    "n_estimators": [100],
    "max_depth": [None],
    "tfidf_max_features": [3000],
    "tfidf_ngram_range": [(1, 1), (1, 2), (2, 3)],
    "tfidf_min_df": [2, 5],
}

In [8]:
results = mod.grid_search_random_forest(df_train, param_grid=param_grid)
results

6 combinations to test...
RMSE: 154,398,432 (+/- 144,366,209) | {'n_estimators': 100, 'max_depth': None, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 2}
RMSE: 214,131,961 (+/- 150,125,838) | {'n_estimators': 100, 'max_depth': None, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 5}
RMSE: 158,440,675 (+/- 142,840,650) | {'n_estimators': 100, 'max_depth': None, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 2}
RMSE: 214,965,455 (+/- 152,142,482) | {'n_estimators': 100, 'max_depth': None, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 5}
RMSE: 162,743,306 (+/- 136,601,695) | {'n_estimators': 100, 'max_depth': None, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (2, 3), 'tfidf_min_df': 2}
RMSE: nan (+/- nan) | {'n_estimators': 100, 'max_depth': None, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (2, 3), 'tfidf_min_df': 5}


/opt/python/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:490: FitFailedWarning: 
4 fits failed out of a total of 10.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/python/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/python/lib/python3.13/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/opt/python/lib/python3.13/site-packages/sklearn/pipeline.py", line 613, in fit
    Xt = self._fit(X, y, routed_params,

,n_estimators,max_depth,tfidf_max_features,tfidf_ngram_range,tfidf_min_df,rmse_mean,rmse_std
0,100,None,3000,"(1, 1)",2,1.543984e+08,1.443662e+08
2,100,None,3000,"(1, 2)",2,1.584407e+08,1.428406e+08
4,100,None,3000,"(2, 3)",2,1.627433e+08,1.366017e+08
1,100,None,3000,"(1, 1)",5,2.141320e+08,1.501258e+08
3,100,None,3000,"(1, 2)",5,2.149655e+08,1.521425e+08
5,100,None,3000,"(2, 3)",5,NaN,NaN


In [9]:
results = mod.grid_search_elastic_net(df_train)

576 combinations to test...
RMSE: 179,911,342 (+/- 120,999,972) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 1}
RMSE: 181,315,478 (+/- 123,192,386) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 2}
RMSE: 201,857,717 (+/- 104,395,917) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 5}
RMSE: 178,987,592 (+/- 122,715,764) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 1}
RMSE: 181,758,435 (+/- 123,331,031) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 2}
RMSE: 201,798,740 (+/- 104,358,534) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 5}
RMSE: 179,161,051 (+/- 120,709,138) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features

/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.310e+16, tolerance: 6.165e+14
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.681e+16, tolerance: 6.250e+14
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.607e+16, 

RMSE: 290,900,414 (+/- 201,088,684) | {'alpha': 0.01, 'l1_ratio': 1.0, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 1}
RMSE: 310,988,612 (+/- 170,979,787) | {'alpha': 0.01, 'l1_ratio': 1.0, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 2}
RMSE: 357,759,549 (+/- 104,809,673) | {'alpha': 0.01, 'l1_ratio': 1.0, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 5}


/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.380e+16, tolerance: 6.291e+14
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.394e+16, tolerance: 6.165e+14
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.674e+16, 

RMSE: 283,819,292 (+/- 279,084,663) | {'alpha': 0.01, 'l1_ratio': 1.0, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 1}
RMSE: 312,977,609 (+/- 161,666,175) | {'alpha': 0.01, 'l1_ratio': 1.0, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 2}
RMSE: 360,844,453 (+/- 105,456,560) | {'alpha': 0.01, 'l1_ratio': 1.0, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 5}


...

/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.394e+16, tolerance: 6.165e+14
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.422e+16, tolerance: 4.218e+14
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.379e+16, 

RMSE: 357,752,463 (+/- 104,813,107) | {'alpha': 10.0, 'l1_ratio': 1.0, 'tfidf_max_features': 5000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 5}
RMSE: 283,816,921 (+/- 279,086,168) | {'alpha': 10.0, 'l1_ratio': 1.0, 'tfidf_max_features': 10000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 1}
RMSE: 312,966,659 (+/- 161,670,106) | {'alpha': 10.0, 'l1_ratio': 1.0, 'tfidf_max_features': 10000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 2}


/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.311e+16, tolerance: 6.165e+14
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.396e+16, tolerance: 6.291e+14
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.422e+16, 

RMSE: 360,837,609 (+/- 105,460,466) | {'alpha': 10.0, 'l1_ratio': 1.0, 'tfidf_max_features': 10000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 5}
RMSE: 302,593,833 (+/- 315,403,339) | {'alpha': 10.0, 'l1_ratio': 1.0, 'tfidf_max_features': 10000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 1}
RMSE: 310,976,404 (+/- 170,983,351) | {'alpha': 10.0, 'l1_ratio': 1.0, 'tfidf_max_features': 10000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 2}
RMSE: 357,752,463 (+/- 104,813,107) | {'alpha': 10.0, 'l1_ratio': 1.0, 'tfidf_max_features': 10000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 5}


/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.379e+16, tolerance: 6.291e+14
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.422e+16, tolerance: 4.218e+14
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.390e+16, 

In [10]:
results.sort_values("rmse_mean")

,alpha,l1_ratio,tfidf_max_features,tfidf_ngram_range,tfidf_min_df,rmse_mean,rmse_std
512,10.00,0.7,3000,"(1, 1)",5,1.461418e+08,1.453085e+08
518,10.00,0.7,5000,"(1, 1)",5,1.461418e+08,1.453085e+08
524,10.00,0.7,10000,"(1, 1)",5,1.461418e+08,1.453085e+08
506,10.00,0.7,1000,"(1, 1)",5,1.461418e+08,1.453085e+08
521,10.00,0.7,5000,"(1, 2)",5,1.461427e+08,1.453081e+08
...,...,...,...,...,...,...,...
266,0.10,1.0,1000,"(1, 1)",5,3.608444e+08,1.054566e+08
140,0.01,1.0,10000,"(1, 1)",5,3.608445e+08,1.054566e+08
128,0.01,1.0,3000,"(1, 1)",5,3.608445e+08,1.054566e+08
122,0.01,1.0,1000,"(1, 1)",5,3.608445e+08,1.054566e+08


In [81]:
param_grid = {
    "alpha": [0.01, 0.1, 1.0],
    "l1_ratio": [0.5, 0.7, 0.9],
    "tfidf_max_features": [5000, 10000],
    "tfidf_ngram_range": [(1, 1), (1, 2)],
    "tfidf_min_df": [2, 5, 10, 20, 30],
    "tfidf_max_df": [0.6, 0.8, 0.9],
}

In [82]:
results_new = mod.grid_search_elastic_net(df_train, param_grid)

540 combinaisons à tester...
RMSE: 146,919,867 (+/- 59,773,628) | {'alpha': 0.01, 'l1_ratio': 0.5, 'tfidf_max_features': 5000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 2, 'tfidf_max_df': 0.6}
RMSE: 146,919,867 (+/- 59,773,628) | {'alpha': 0.01, 'l1_ratio': 0.5, 'tfidf_max_features': 5000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 2, 'tfidf_max_df': 0.8}
RMSE: 146,919,867 (+/- 59,773,628) | {'alpha': 0.01, 'l1_ratio': 0.5, 'tfidf_max_features': 5000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 2, 'tfidf_max_df': 0.9}
RMSE: 148,003,460 (+/- 58,291,009) | {'alpha': 0.01, 'l1_ratio': 0.5, 'tfidf_max_features': 5000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 5, 'tfidf_max_df': 0.6}
RMSE: 148,003,460 (+/- 58,291,009) | {'alpha': 0.01, 'l1_ratio': 0.5, 'tfidf_max_features': 5000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 5, 'tfidf_max_df': 0.8}
RMSE: 148,003,460 (+/- 58,291,009) | {'alpha': 0.01, 'l1_ratio': 0.5, 'tfidf_max_features': 5000, 'tfidf_ngram_range': (1, 1), 'tfidf_mi

In [83]:
results_new.sort_values(["rmse_mean"]).head(20)

,alpha,l1_ratio,tfidf_max_features,tfidf_ngram_range,tfidf_min_df,tfidf_max_df,rmse_mean,rmse_std
266,0.1,0.7,5000,"(1, 2)",20,0.9,1.428925e+08,6.302261e+07
264,0.1,0.7,5000,"(1, 2)",20,0.6,1.428925e+08,6.302261e+07
265,0.1,0.7,5000,"(1, 2)",20,0.8,1.428925e+08,6.302261e+07
250,0.1,0.7,5000,"(1, 1)",20,0.8,1.428925e+08,6.302261e+07
251,0.1,0.7,5000,"(1, 1)",20,0.9,1.428925e+08,6.302261e+07
249,0.1,0.7,5000,"(1, 1)",20,0.6,1.428925e+08,6.302261e+07
280,0.1,0.7,10000,"(1, 1)",20,0.8,1.428925e+08,6.302261e+07
279,0.1,0.7,10000,"(1, 1)",20,0.6,1.428925e+08,6.302261e+07
281,0.1,0.7,10000,"(1, 1)",20,0.9,1.428925e+08,6.302261e+07
295,0.1,0.7,10000,"(1, 2)",20,0.8,1.428925e+08,6.302261e+07
